RAG PIPELINE -Data Ingestion to vector DB pipelines

In [9]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")
print(all_pdf_documents)

Found 2 PDF files to process

Processing: Context.pdf
  ✓ Loaded 23 pages

Processing: OOP notes.pdf
  ✓ Loaded 20 pages

Total documents loaded: 43
[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2025-10-29T17:50:14+01:00', 'moddate': '2025-11-03T13:21:51+01:00', 'source': '..\\data\\pdf\\Context.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Context.pdf', 'file_type': 'pdf'}, page_content='Context\nEngineering\nDesigning the systems that control what \ninformation reaches the model and how it \nmaintains coherence.'), Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2025-10-29T17:50:14+01:00', 'moddate': '2025-11-03T13:21:51+01:00', 'source': '..\\data\\pdf\\Context.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'Context.pdf', 'file_type': 'pdf'}, page_content='Introduction\nThe Ev olution: F r om Pr ompt s t o A ctions\nThe Or chestr ation Challenge\nThe Ne xt F r ontier of T ool

In [13]:

# ------------------------------------------------
# SPLIT DOCUMENTS INTO CHUNKS
# ------------------------------------------------
def chunk_splitting(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"\nSplit {len(documents)} documents into {len(split_docs)} chunks")

    # show example chunk
    if split_docs:
        print("\nExample Chunk:")
        print(split_docs[0].page_content[:200])
        print("\nMetadata:", split_docs[0].metadata)

    return split_docs


# ------------------------------------------------
# MAIN PIPELINE
# ------------------------------------------------
if __name__ == "__main__":

    pdf_directory = "../data"

    # Step 1: Load PDFs
    documents = process_read_pdf(pdf_directory)

    # Step 2: Split into chunks
    chunked_data = chunk_splitting(documents)

    print(f"\nTotal chunks created: {len(chunked_data)}")


Processing 2 PDF files...

Total documents loaded: 43

Split 43 documents into 83 chunks

Example Chunk:
Context
Engineering
Designing the systems that control what 
information reaches the model and how it 
maintains coherence.

Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2025-10-29T17:50:14+01:00', 'moddate': '2025-11-03T13:21:51+01:00', 'source': '..\\data\\pdf\\Context.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Context.pdf', 'file_type': 'pdf'}

Total chunks created: 83
